In [1]:
!pip install faiss-cpu


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 68.9 MB/s eta 0:00:00


In [2]:
# ============================================================================
# CELL 1: IMPORTS & DEPENDENCIES
# ============================================================================
"""
Part 1A: Import all required libraries for the historical NLP pipeline.
- Data processing: pandas, numpy
- NLP: transformers, sentence-transformers
- Vector DB: FAISS
- Generation: HuggingFace pipeline
"""

import pandas as pd
import numpy as np
import re
import torch
import faiss
import textwrap
import warnings
from typing import Dict, List, Tuple, Any, Optional
from collections import defaultdict
from dataclasses import dataclass, field

# Transformers
from transformers import (
    AutoTokenizer, 
    AutoModel, 
    AutoModelForCausalLM,
    pipeline,
    BitsAndBytesConfig
)
from sentence_transformers import SentenceTransformer, util

warnings.filterwarnings('ignore')

print("✅ All imports successful")
print(f"🔧 PyTorch version: {torch.__version__}")
print(f"🔧 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🔧 GPU: {torch.cuda.get_device_name(0)}")

2026-02-10 16:38:01.510133: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770741481.699945      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770741481.761033      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770741482.240921      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770741482.240965      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770741482.240968      24 computation_placer.cc:177] computation placer alr

✅ All imports successful
🔧 PyTorch version: 2.8.0+cu126
🔧 CUDA available: True
🔧 GPU: Tesla T4


In [3]:
# ============================================================================
# CELL 2: LOAD TRANSFORMER MODELS
# ============================================================================
"""
Part 1B: Load Historical BERT for entity extraction and contextual understanding,
plus a sentence transformer for semantic embeddings.

Model hierarchy:
  1. dbmdz/bert-base-historic-english-cased (primary - trained on historical text)
  2. roberta-base (fallback)

Embedding model:
  - all-mpnet-base-v2 (768-dim, best quality)
  - all-MiniLM-L6-v2 (384-dim, fallback)
"""

print("=" * 60)
print("🔄 LOADING TRANSFORMER MODELS")
print("=" * 60)

# --- Historical BERT ---
HISTORICAL_BERT = "dbmdz/bert-base-historic-english-cased"

try:
    print(f"\n📥 Loading {HISTORICAL_BERT}...")
    hist_tokenizer = AutoTokenizer.from_pretrained(HISTORICAL_BERT)
    hist_model = AutoModel.from_pretrained(HISTORICAL_BERT)
    hist_model.eval()
    USE_HISTORICAL = True
    print("   ✅ Historical BERT loaded successfully")
except Exception as e:
    print(f"   ⚠️ Could not load historical model: {e}")
    print(f"   📥 Falling back to roberta-base...")
    hist_tokenizer = AutoTokenizer.from_pretrained("roberta-base")
    hist_model = AutoModel.from_pretrained("roberta-base")
    hist_model.eval()
    USE_HISTORICAL = False

# --- Sentence Transformer for Embeddings ---
EMBED_MODEL = "all-mpnet-base-v2" if USE_HISTORICAL else "all-MiniLM-L6-v2"
print(f"\n📥 Loading sentence embedder: {EMBED_MODEL}...")
embedder = SentenceTransformer(EMBED_MODEL)
EMBED_DIM = embedder.get_sentence_embedding_dimension()
print(f"   ✅ Embedder loaded | Dimension: {EMBED_DIM}")

# --- Move to GPU if available ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
hist_model = hist_model.to(device)
print(f"\n🖥️  Device: {device}")
print("=" * 60)

🔄 LOADING TRANSFORMER MODELS

📥 Loading dbmdz/bert-base-historic-english-cased...


tokenizer_config.json:   0%|          | 0.00/392 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

   ✅ Embedder loaded | Dimension: 768

🖥️  Device: cuda


In [4]:
# ============================================================================
# CELL 3: HISTORICAL TEXT CLEANER
# ============================================================================
"""
Part 2A: HistoricalTextCleaner — Domain-specific text normalization.

Handles:
  - Date normalization (BC/BCE/AD/CE, centuries, circa dates)
  - Historical name resolution (variant → canonical form)
  - Historical term recognition
  - Period and civilization detection
"""

class HistoricalTextCleaner:
    """Cleans and normalizes historical text with domain-specific rules."""
    
    def __init__(self):
        # --- Date Normalization Patterns ---
        self.date_patterns = [
            (r'\b(\d+)\s*(BC|BCE|B\.C\.|B\.C\.E\.)\b', r'\1 BCE'),
            (r'\b(\d+)\s*(AD|CE|A\.D\.|C\.E\.)\b', r'\1 CE'),
            (r'\bcirca\s*(\d+)\s*(?:BCE?|CE)?\b', r'circa \1'),
            (r'\b(\d+)\s*-\s*(\d+)\s*(?:century|centuries)\s*(BCE?)?\b', 
             self._format_century_range),
            (r'\b(\d+)(?:st|nd|rd|th)\s+century\s*(BCE?)?\b', 
             self._century_to_years),
            (r'\b(\d+)\s*/\s*(\d+)\s*(?:BCE?|CE)?\b', r'\1-\2'),
        ]
        
        # --- Historical Name Variants → Canonical ---
        self.historical_names = {
            'hannibal barca': [
                'hannibal', 'barca', 'hannibal of carthage',
                'the great hannibal', 'carthaginian general hannibal'
            ],
            'scipio africanus': [
                'scipio', 'publius cornelius scipio', 
                'scipio the elder', 'africanus'
            ],
            'julius caesar': [
                'caesar', 'gaius julius caesar', 'dictator caesar'
            ],
            'habib bourguiba': [
                'bourguiba', 'habib bourguiba', 'president bourguiba'
            ],
            'queen dido': [
                'dido', 'elissa', 'princess elissa', 
                'founder of carthage', 'dido of carthage'
            ],
            'gaiseric': [
                'genseric', 'king gaiseric', 'gaiseric the vandal',
                'vandal king gaiseric'
            ],
            'ibn khaldun': [
                'abd al-rahman ibn khaldun', 'khaldun',
                'the great historian ibn khaldun'
            ],
            'alexander the great': [
                'alexander', 'alexander of macedon', 'king alexander'
            ],
            'cleopatra': [
                'cleopatra vii', 'queen cleopatra', 'cleopatra of egypt'
            ],
            'saladin': [
                'salah ad-din', 'salah al-din', 'sultan saladin'
            ],
            'augustine of hippo': [
                'saint augustine', 'augustine', 'st augustine'
            ],
            'massinissa': [
                'masinissa', 'king massinissa', 'numidian king massinissa'
            ],
            'jugurtha': [
                'king jugurtha', 'jugurtha of numidia'
            ],
        }
        
        # --- Historical Domain Terms ---
        self.historical_terms = {
            'punic', 'carthaginian', 'roman', 'byzantine', 'vandal', 'arab',
            'ottoman', 'hafsid', 'fatimid', 'aghlabid', 'almohad', 'almoravid',
            'berber', 'numidian', 'phoenician', 'islamic', 'crusader',
            'hellenistic', 'ptolemaic', 'sassanid', 'umayyad', 'abbasid',
            'mamluk', 'zirid', 'hammadid', 'rustamid', 'idrisid'
        }
    
    def _format_century_range(self, match):
        start, end, era = match.groups()
        era_str = f" {era}" if era else ""
        return f"{start}-{end} century{era_str}"
    
    def _century_to_years(self, match):
        century, era = match.groups()
        century_num = int(century)
        start_year = (century_num - 1) * 100 + 1
        end_year = century_num * 100
        era_str = f" {era}" if era else " CE"
        return f"{start_year}-{end_year}{era_str}"
    
    def normalize_dates(self, text: str) -> str:
        """Standardize all date formats in the text."""
        for pattern, replacement in self.date_patterns:
            if callable(replacement):
                text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
            else:
                text = re.sub(pattern, replacement, text, flags=re.IGNORECASE)
        return text
    
    def normalize_names(self, text: str) -> str:
        """Resolve name variants to canonical forms."""
        text_lower = text.lower()
        for canonical, variants in self.historical_names.items():
            for variant in sorted(variants, key=len, reverse=True):
                if variant.lower() in text_lower:
                    pattern = re.compile(re.escape(variant), re.IGNORECASE)
                    text = pattern.sub(canonical, text)
                    text_lower = text.lower()
        return text
    
    def clean_text(self, text: str) -> str:
        """Full cleaning pipeline for historical text."""
        if pd.isna(text) or not text:
            return ""
        
        text = str(text).strip()
        text = self.normalize_dates(text)
        text = self.normalize_names(text)
        
        # Lowercase
        text = text.lower()
        
        # Remove special characters but keep hyphens in dates
        text = re.sub(r'[^\w\s\-]', ' ', text)
        
        # Collapse whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        return text
    
    def extract_historical_context(self, text: str) -> Dict[str, Any]:
        """Extract structured historical metadata from text."""
        if not text:
            return {'time_period': None, 'civilization': None, 'key_figures': []}
        
        context = {
            'time_period': None,
            'civilization': None,
            'key_figures': []
        }
        
        text_lower = text.lower() if text else ""
        
        # --- Time Period Detection ---
        period_patterns = {
            'prehistoric': r'\b(prehistoric|stone age|neolithic|paleolithic)\b',
            'ancient': r'\b(ancient|classical|antiquity|bce|bc)\b',
            'late antiquity': r'\b(late antiquity|fall of rome|vandal|byzantine)\b',
            'medieval': r'\b(medieval|middle ages|dark ages|hafsid|aghlabid)\b',
            'early modern': r'\b(renaissance|reformation|early modern|ottoman)\b',
            'colonial': r'\b(colonial|protectorate|french rule)\b',
            'modern': r'\b(modern|contemporary|independence|republic)\b'
        }
        
        for period, pattern in period_patterns.items():
            if re.search(pattern, text_lower):
                context['time_period'] = period
                break
        
        # --- Civilization Detection ---
        civ_patterns = {
            'phoenician': r'\b(phoenician|tyre|sidon)\b',
            'carthaginian': r'\b(carthaginian|punic|carthage)\b',
            'numidian': r'\b(numidian|numidia|berber|amazigh)\b',
            'roman': r'\b(roman|rome|republic|empire|latin)\b',
            'vandal': r'\b(vandal|gaiseric|genseric)\b',
            'byzantine': r'\b(byzantine|eastern roman|constantinople)\b',
            'arab-islamic': r'\b(islamic|muslim|arab|umayyad|abbasid|fatimid)\b',
            'ottoman': r'\b(ottoman|turk|sultan|bey)\b',
            'french colonial': r'\b(french|colonial|protectorate)\b',
            'tunisian': r'\b(tunisian|tunisia|bourguiba|independence)\b'
        }
        
        for civ, pattern in civ_patterns.items():
            if re.search(pattern, text_lower):
                context['civilization'] = civ
                break
        
        # --- Key Figure Detection ---
        for canonical in self.historical_names:
            if canonical.lower() in text_lower:
                context['key_figures'].append(canonical)
        
        return context


# --- Instantiate ---
cleaner = HistoricalTextCleaner()

# Quick test
test_text = "Hannibal crossed the Alps in 218 BC to fight Scipio at the Battle of Zama"
print("🧪 Cleaner Test:")
print(f"   Input:  {test_text}")
print(f"   Output: {cleaner.clean_text(test_text)}")
print(f"   Context: {cleaner.extract_historical_context(test_text)}")

🧪 Cleaner Test:
   Input:  Hannibal crossed the Alps in 218 BC to fight Scipio at the Battle of Zama
   Output: hannibal hannibal barca crossed the alps in 218 bce to fight scipio africanus at the battle of zama
   Context: {'time_period': 'ancient', 'civilization': None, 'key_figures': []}


In [5]:
# ============================================================================
# CELL 4: HISTORICAL ENTITY EXTRACTOR
# ============================================================================
"""
Part 2B: HistoricalEntityExtractor — Uses Historical BERT for:
  - Token-level entity extraction with type classification
  - Document-level embedding generation (CLS / mean / max pooling)
  
Entity Types: PERSON, LOCATION, DATE, EVENT, CIVILIZATION, OTHER
"""

class HistoricalEntityExtractor:
    """Extract historical entities and create embeddings using BERT."""
    
    def __init__(self, tokenizer, model, device='cpu'):
        self.tokenizer = tokenizer
        self.model = model
        self.device = device
        
        # --- Entity Type Lexicons ---
        self.entity_lexicons = {
            'PERSON': {
                'hannibal', 'scipio', 'caesar', 'bourguiba', 'dido', 'elissa',
                'gaiseric', 'alexander', 'napoleon', 'cleopatra', 'saladin',
                'augustine', 'massinissa', 'jugurtha', 'ibn khaldun', 'khaldun',
                'king', 'queen', 'emperor', 'sultan', 'pharaoh', 'caliph',
                'ruler', 'leader', 'general', 'commander', 'consul', 'senator',
                'imam', 'sheikh', 'bey', 'dey', 'pasha'
            },
            'LOCATION': {
                'carthage', 'rome', 'tunis', 'alps', 'zama', 'mediterranean',
                'africa', 'europe', 'asia', 'kairouan', 'mahdia', 'sousse',
                'sfax', 'bizerte', 'dougga', 'el jem', 'kerkouane', 'utica',
                'hippo', 'thapsus', 'hadrumetum', 'leptis', 'numidia',
                'ifriqiya', 'maghreb', 'iberia', 'sicily', 'sardinia',
                'city', 'empire', 'kingdom', 'province', 'river', 'sea',
                'mountain', 'desert', 'coast', 'port', 'peninsula'
            },
            'DATE': {
                'bc', 'bce', 'ad', 'ce', 'century', 'year', 'period', 'era',
                'age', 'ancient', 'medieval', 'modern', 'decade', 'millennium',
                'circa', 'dynasty', 'reign', 'epoch'
            },
            'EVENT': {
                'battle', 'war', 'siege', 'treaty', 'revolution', 'conquest',
                'invasion', 'campaign', 'revolt', 'uprising', 'rebellion',
                'crusade', 'expedition', 'conflict', 'attack', 'defense',
                'founding', 'destruction', 'fall', 'rise', 'independence',
                'colonization', 'liberation', 'migration', 'trade'
            },
            'CIVILIZATION': {
                'punic', 'carthaginian', 'roman', 'byzantine', 'vandal',
                'phoenician', 'numidian', 'berber', 'arab', 'islamic',
                'ottoman', 'french', 'hafsid', 'fatimid', 'aghlabid',
                'almohad', 'almoravid', 'zirid', 'umayyad', 'abbasid'
            }
        }
    
    def extract_entities(self, text: str, max_length: int = 512) -> List[Dict]:
        """Extract named entities using BERT token analysis."""
        if not text or not text.strip():
            return []
        
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=max_length
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        token_embeddings = outputs.last_hidden_state[0]
        tokens = self.tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        
        # Reconstruct words from subword tokens
        entities = []
        current_word = []
        current_indices = []
        
        special_tokens = {'[CLS]', '[SEP]', '[PAD]', '[UNK]', 
                         '<s>', '</s>', '<pad>', '<unk>'}
        
        for i, token in enumerate(tokens):
            if token in special_tokens:
                if current_word:
                    self._finalize_entity(
                        current_word, current_indices, 
                        token_embeddings, entities
                    )
                    current_word = []
                    current_indices = []
                continue
            
            # Handle subword tokens
            if token.startswith('##') or token.startswith('Ġ'):
                clean_token = token.lstrip('#').lstrip('Ġ')
                current_word.append(clean_token)
                current_indices.append(i)
            else:
                # New word boundary
                if current_word:
                    self._finalize_entity(
                        current_word, current_indices, 
                        token_embeddings, entities
                    )
                current_word = [token]
                current_indices = [i]
        
        # Don't forget last word
        if current_word:
            self._finalize_entity(
                current_word, current_indices, 
                token_embeddings, entities
            )
        
        # Filter out noise (single chars, pure numbers, stopwords)
        stopwords = {'the', 'a', 'an', 'in', 'on', 'at', 'to', 'of', 'and',
                     'or', 'but', 'is', 'was', 'were', 'are', 'been', 'be',
                     'has', 'had', 'have', 'do', 'did', 'does', 'will',
                     'with', 'for', 'from', 'by', 'as', 'it', 'its',
                     'this', 'that', 'these', 'those', 'he', 'she', 'they',
                     'his', 'her', 'their', 'who', 'which', 'what', 'when',
                     'where', 'how', 'not', 'no', 'if', 'than', 'then'}
        
        filtered = []
        for ent in entities:
            text_clean = ent['text'].strip()
            if (len(text_clean) > 1 and 
                text_clean.lower() not in stopwords and
                not text_clean.isdigit()):
                ent['text'] = text_clean
                filtered.append(ent)
        
        return filtered
    
    def _finalize_entity(self, word_parts, indices, embeddings, entities_list):
        """Combine subword tokens into a complete entity."""
        word = ''.join(word_parts)
        word = re.sub(r'[^\w\s\-]', '', word).strip()
        
        if word:
            # Average embedding of all subword tokens
            emb = torch.stack([embeddings[i] for i in indices]).mean(dim=0)
            entity_type = self._classify_entity(word)
            
            entities_list.append({
                'text': word,
                'type': entity_type,
                'embedding': emb.cpu().numpy()
            })
    
    def _classify_entity(self, token: str) -> str:
        """Classify entity type using lexicon matching."""
        token_lower = token.lower()
        
        for entity_type, lexicon in self.entity_lexicons.items():
            if any(term in token_lower for term in lexicon):
                return entity_type
        
        # Heuristic: capitalized words are likely proper nouns
        if token[0].isupper():
            return 'PROPER_NOUN'
        
        return 'OTHER'
    
    def create_embedding(self, text: str, method: str = 'mean') -> np.ndarray:
        """Create a document-level embedding using Historical BERT.
        
        Args:
            text: Input text
            method: Pooling strategy - 'cls', 'mean', or 'max'
        
        Returns:
            numpy array of shape (hidden_dim,)
        """
        if not text or not text.strip():
            # Return zero vector
            return np.zeros(self.model.config.hidden_size)
        
        inputs = self.tokenizer(
            text,
            return_tensors='pt',
            truncation=True,
            padding=True,
            max_length=512
        ).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(**inputs)
        
        hidden_states = outputs.last_hidden_state[0]  # (seq_len, hidden_dim)
        
        # Mask out padding tokens
        attention_mask = inputs['attention_mask'][0].unsqueeze(-1).float()
        
        if method == 'cls':
            embedding = hidden_states[0]
        elif method == 'mean':
            masked = hidden_states * attention_mask
            embedding = masked.sum(dim=0) / attention_mask.sum(dim=0).clamp(min=1)
        elif method == 'max':
            masked = hidden_states.clone()
            masked[attention_mask.squeeze(-1) == 0] = -1e9
            embedding = masked.max(dim=0)[0]
        else:
            embedding = hidden_states[0]
        
        return embedding.cpu().numpy()


# --- Instantiate ---
extractor = HistoricalEntityExtractor(
    hist_tokenizer, hist_model, 
    device=str(device)
)

# Quick test
test_entities = extractor.extract_entities(
    "Hannibal Barca crossed the Alps in 218 BCE to attack Rome"
)
print("🧪 Entity Extraction Test:")
for ent in test_entities:
    print(f"   → {ent['text']:20s} | Type: {ent['type']}")

🧪 Entity Extraction Test:
   → crossed              | Type: OTHER
   → attack               | Type: EVENT


In [6]:
# ============================================================================
# CELL 5: LOAD DATASET & INSPECT
# ============================================================================
"""
Part 3A: Load the historical events dataset and perform initial inspection.
"""

DATASET_PATH = "/kaggle/input/history-of-tounes/historical_events_cleaned_full (1).csv"

print("=" * 60)
print("📊 LOADING HISTORICAL DATASET")
print("=" * 60)

try:
    df = pd.read_csv(DATASET_PATH, encoding='utf-8')
    print(f"\n✅ Dataset loaded successfully")
    print(f"   Rows: {len(df)}")
    print(f"   Columns: {len(df.columns)}")
    print(f"\n📋 Column names:")
    for i, col in enumerate(df.columns, 1):
        null_count = df[col].isna().sum()
        print(f"   {i:2d}. {col:30s} | nulls: {null_count} | dtype: {df[col].dtype}")
    
    print(f"\n📝 First 3 rows preview:")
    display(df.head(3))
    
except Exception as e:
    print(f"❌ Error loading dataset: {e}")
    raise

📊 LOADING HISTORICAL DATASET

✅ Dataset loaded successfully
   Rows: 114
   Columns: 34

📋 Column names:
    1. event_id                       | nulls: 0 | dtype: object
    2. name_of_incident               | nulls: 0 | dtype: object
    3. year                           | nulls: 0 | dtype: int64
    4. country                        | nulls: 0 | dtype: object
    5. place_name                     | nulls: 0 | dtype: object
    6. type_of_event                  | nulls: 0 | dtype: object
    7. historical_character           | nulls: 0 | dtype: object
    8. description                    | nulls: 0 | dtype: object
    9. first_person_strategy          | nulls: 0 | dtype: object
   10. mistakes_reflection            | nulls: 0 | dtype: object
   11. modern_strategy                | nulls: 0 | dtype: object
   12. description_quality            | nulls: 0 | dtype: object
   13. first_person_strategy_quality  | nulls: 0 | dtype: object
   14. mistakes_reflection_quality    | nulls: 0 | 

,event_id,name_of_incident,year,country,place_name,type_of_event,historical_character,description,first_person_strategy,mistakes_reflection,...,mistakes_reflection_char_count,modern_strategy_word_count,modern_strategy_char_count,total_narrative_words,narrative_completeness,narrative_completeness_pct,rag_document,rag_document_word_count,first_person_question_count,first_person_exclamation_count
0,EVT_00001,Crossing the Alps,-218,Tunisia,Alps Mountains,Military Campaign,Hannibal Barca,"Led 50,000 infantry, 9,000 cavalry, and 37 war...",I knew Rome controlled the seas after our defe...,My greatest mistake was not marching on Rome i...,...,193,37,250,148,4,100.0,Event: Crossing the Alps | Year: 218 BCE | Loc...,181,0,0
1,EVT_00002,Battle of Cannae,-216,Tunisia,"Cannae, Apulia",Battle,Hannibal Barca,"My greatest tactical victory. With 50,000 men,...","I used my weakest troops in the center, delibe...","After this crushing victory, I still didn't ma...",...,214,34,232,154,4,100.0,Event: Battle of Cannae | Year: 216 BCE | Loca...,186,0,0
2,EVT_00003,Lack of Reinforcements from Carthage,-215,Tunisia,Carthage Senate,Political Decision,Hannibal Barca,"The Carthaginian Senate, led by the Hanno fact...",My strategy required sustained pressure on Rom...,I relied too heavily on support from home. I s...,...,227,37,251,146,4,100.0,Event: Lack of Reinforcements from Carthage | ...,181,0,0


In [7]:
# ============================================================================
# CELL 6: RAG DOCUMENT CHUNKING
# ============================================================================
"""
Part 3B: Transform raw dataset rows into RAG-ready document chunks.

Strategy:
  - Each row becomes a primary document
  - Long descriptions are split into overlapping chunks
  - Each chunk retains metadata (event, character, year, civilization)
  - Section-aware splitting respects sentence boundaries
"""

@dataclass
class HistoricalDocument:
    """A single RAG-ready document chunk with metadata."""
    doc_id: str
    text: str                    # cleaned text
    raw_text: str                # original text
    embedding: Optional[np.ndarray] = None
    metadata: Dict = field(default_factory=dict)
    

def create_rag_documents(
    df: pd.DataFrame, 
    cleaner: HistoricalTextCleaner,
    chunk_size: int = 300,       # words per chunk
    chunk_overlap: int = 50,     # overlap in words
) -> List[HistoricalDocument]:
    """
    Convert DataFrame rows into overlapping, metadata-enriched chunks.
    
    Args:
        df: Historical events DataFrame
        cleaner: Text cleaner instance
        chunk_size: Max words per chunk
        chunk_overlap: Word overlap between consecutive chunks
    
    Returns:
        List of HistoricalDocument objects
    """
    documents = []
    doc_counter = 0
    
    # Detect available columns
    text_cols = [c for c in ['description', 'details', 'summary', 'text'] 
                 if c in df.columns]
    name_col = next((c for c in ['name_of_incident', 'event_name', 'name', 'title'] 
                     if c in df.columns), None)
    char_col = next((c for c in ['historical_character', 'character', 'figure', 'person'] 
                     if c in df.columns), None)
    year_col = next((c for c in ['year', 'date', 'period'] 
                     if c in df.columns), None)
    
    print(f"📄 Text columns: {text_cols}")
    print(f"📄 Name column: {name_col}")
    print(f"📄 Character column: {char_col}")
    print(f"📄 Year column: {year_col}")
    
    for idx, row in df.iterrows():
        # --- Build raw text ---
        parts = []
        
        event_name = str(row.get(name_col, '')) if name_col else ''
        character = str(row.get(char_col, 'Unknown')) if char_col else 'Unknown'
        year = str(row.get(year_col, 'Unknown')) if year_col else 'Unknown'
        
        if event_name and event_name != 'nan':
            parts.append(f"Event: {event_name}.")
        
        for tc in text_cols:
            val = str(row.get(tc, ''))
            if val and val != 'nan':
                parts.append(val)
        
        # Add any remaining string columns as supplementary context
        for col in df.columns:
            if col not in text_cols and col not in [name_col, char_col, year_col]:
                val = str(row.get(col, ''))
                if val and val != 'nan' and len(val) > 20:
                    parts.append(val)
        
        raw_text = ' '.join(parts).strip()
        if not raw_text:
            continue
        
        # --- Clean text ---
        cleaned = cleaner.clean_text(raw_text)
        if not cleaned:
            continue
        
        # --- Extract context ---
        context = cleaner.extract_historical_context(raw_text)
        
        # --- Metadata ---
        meta = {
            'event_name': event_name if event_name != 'nan' else 'Unknown',
            'character': character if character != 'nan' else 'Unknown',
            'year': year if year != 'nan' else 'Unknown',
            'row_index': idx,
            'time_period': context.get('time_period'),
            'civilization': context.get('civilization'),
            'key_figures': context.get('key_figures', []),
        }
        
        # --- Chunk the text ---
        words = cleaned.split()
        
        if len(words) <= chunk_size:
            # Single chunk
            doc = HistoricalDocument(
                doc_id=f"doc_{doc_counter:04d}",
                text=cleaned,
                raw_text=raw_text[:500],
                metadata=meta
            )
            documents.append(doc)
            doc_counter += 1
        else:
            # Multiple overlapping chunks
            start = 0
            chunk_idx = 0
            while start < len(words):
                end = min(start + chunk_size, len(words))
                chunk_text = ' '.join(words[start:end])
                
                chunk_meta = meta.copy()
                chunk_meta['chunk_index'] = chunk_idx
                chunk_meta['total_chunks'] = -1  # will update
                
                doc = HistoricalDocument(
                    doc_id=f"doc_{doc_counter:04d}_chunk_{chunk_idx}",
                    text=chunk_text,
                    raw_text=raw_text[:500],
                    metadata=chunk_meta
                )
                documents.append(doc)
                doc_counter += 1
                chunk_idx += 1
                
                start += chunk_size - chunk_overlap
                if end >= len(words):
                    break
            
            # Update total_chunks count
            for d in documents[-chunk_idx:]:
                d.metadata['total_chunks'] = chunk_idx
    
    print(f"\n✅ Created {len(documents)} document chunks from {len(df)} rows")
    
    # Stats
    text_lengths = [len(d.text.split()) for d in documents]
    print(f"   Avg chunk size: {np.mean(text_lengths):.0f} words")
    print(f"   Min: {np.min(text_lengths)} | Max: {np.max(text_lengths)} words")
    
    # Character distribution
    chars = [d.metadata.get('character', 'Unknown') for d in documents]
    char_counts = pd.Series(chars).value_counts().head(10)
    print(f"\n📊 Top characters in documents:")
    for char, count in char_counts.items():
        print(f"   {char}: {count} chunks")
    
    return documents


# --- Create documents ---
print("=" * 60)
print("📄 CREATING RAG DOCUMENT CHUNKS")
print("=" * 60)

documents = create_rag_documents(
    df, cleaner, 
    chunk_size=250, 
    chunk_overlap=40
)

📄 CREATING RAG DOCUMENT CHUNKS
📄 Text columns: ['description']
📄 Name column: name_of_incident
📄 Character column: historical_character
📄 Year column: year

✅ Created 208 document chunks from 114 rows
   Avg chunk size: 173 words
   Min: 41 | Max: 250 words

📊 Top characters in documents:
   Hannibal Barca: 20 chunks
   Habib Bourguiba: 17 chunks
   Masinissa: 8 chunks
   Zine El Abidine Ben Ali: 7 chunks
   Hanno The Great: 6 chunks
   Hamilcar Barca: 4 chunks
   Ahmed Bey: 4 chunks
   Kais Saied: 4 chunks
   Tunisian Protesters: 4 chunks
   Tunisian Youth: 4 chunks


In [8]:
# ============================================================================
# CELL 7: GENERATE SEMANTIC EMBEDDINGS
# ============================================================================
"""
Part 3C: Generate embeddings for all document chunks using the sentence transformer.
These embeddings will be stored in the FAISS vector index.
"""

print("=" * 60)
print("🔢 GENERATING SEMANTIC EMBEDDINGS")
print("=" * 60)

# Collect all texts
all_texts = [doc.text for doc in documents]

# Batch encode with progress
BATCH_SIZE = 64
all_embeddings = []

print(f"\n📐 Encoding {len(all_texts)} chunks in batches of {BATCH_SIZE}...")
for i in range(0, len(all_texts), BATCH_SIZE):
    batch = all_texts[i:i + BATCH_SIZE]
    batch_emb = embedder.encode(
        batch, 
        convert_to_numpy=True,
        show_progress_bar=False,
        normalize_embeddings=True  # L2 normalize for cosine similarity
    )
    all_embeddings.append(batch_emb)
    
    if (i + BATCH_SIZE) % (BATCH_SIZE * 5) == 0 or i + BATCH_SIZE >= len(all_texts):
        print(f"   Encoded {min(i + BATCH_SIZE, len(all_texts))}/{len(all_texts)}")

# Stack into matrix
embedding_matrix = np.vstack(all_embeddings).astype('float32')

# Assign embeddings back to documents
for i, doc in enumerate(documents):
    doc.embedding = embedding_matrix[i]

print(f"\n✅ Embedding matrix shape: {embedding_matrix.shape}")
print(f"   Dimension: {embedding_matrix.shape[1]}")
print(f"   Normalized: True (L2)")

🔢 GENERATING SEMANTIC EMBEDDINGS

📐 Encoding 208 chunks in batches of 64...
   Encoded 208/208

✅ Embedding matrix shape: (208, 768)
   Dimension: 768
   Normalized: True (L2)


In [9]:
# ============================================================================
# CELL 8: BUILD FAISS VECTOR INDEX
# ============================================================================
"""
Part 4A: Build a FAISS index for fast approximate nearest neighbor search.

Index type: IndexFlatIP (Inner Product) — works as cosine similarity 
since embeddings are L2-normalized.
"""

print("=" * 60)
print("🗄️  BUILDING FAISS VECTOR INDEX")
print("=" * 60)

# Build index
dimension = embedding_matrix.shape[1]
faiss_index = faiss.IndexFlatIP(dimension)  # Inner Product = Cosine (normalized)
faiss_index.add(embedding_matrix)

print(f"\n✅ FAISS index built")
print(f"   Index type: IndexFlatIP (cosine similarity)")
print(f"   Vectors stored: {faiss_index.ntotal}")
print(f"   Dimension: {dimension}")

# Verify with a self-query
test_query = embedder.encode(
    ["battle of carthage"], 
    normalize_embeddings=True
).astype('float32')
scores, indices = faiss_index.search(test_query, 3)

print(f"\n🧪 Verification query: 'battle of carthage'")
for rank, (score, idx) in enumerate(zip(scores[0], indices[0]), 1):
    doc = documents[idx]
    print(f"   {rank}. Score={score:.4f} | {doc.metadata.get('event_name', 'N/A')[:50]}")
    print(f"      Character: {doc.metadata.get('character', 'N/A')}")

🗄️  BUILDING FAISS VECTOR INDEX

✅ FAISS index built
   Index type: IndexFlatIP (cosine similarity)
   Vectors stored: 208
   Dimension: 768

🧪 Verification query: 'battle of carthage'
   1. Score=0.7479 | The Fall of Carthage
      Character: Hannibal Barca
   2. Score=0.7209 | The Fall of Carthage
      Character: Hannibal Barca
   3. Score=0.6596 | Battle of Zama - Final Defeat
      Character: Hannibal Barca


In [10]:
# ============================================================================
# CELL 9: RETRIEVAL ENGINE
# ============================================================================
"""
Part 4B: Retrieval engine that combines FAISS vector search with
optional metadata filtering and re-ranking using Historical BERT.
"""

class HistoricalRetriever:
    """Retrieve relevant historical documents for a given query."""
    
    def __init__(
        self, 
        faiss_index, 
        documents: List[HistoricalDocument],
        embedder: SentenceTransformer,
        extractor: HistoricalEntityExtractor,
        cleaner: HistoricalTextCleaner,
        bert_rerank_weight: float = 0.3
    ):
        self.index = faiss_index
        self.documents = documents
        self.embedder = embedder
        self.extractor = extractor
        self.cleaner = cleaner
        self.bert_weight = bert_rerank_weight
    
    def retrieve(
        self, 
        query: str, 
        top_k: int = 5, 
        use_reranking: bool = True,
        character_filter: Optional[str] = None,
        min_score: float = 0.0
    ) -> List[Tuple[HistoricalDocument, float]]:
        """
        Retrieve top-k relevant documents for a query.
        
        Args:
            query: User's natural language question
            top_k: Number of results to return
            use_reranking: Whether to re-rank with Historical BERT
            character_filter: Optional filter by historical character
            min_score: Minimum relevance threshold
        
        Returns:
            List of (document, score) tuples, sorted by relevance
        """
        # Clean the query
        cleaned_query = self.cleaner.clean_text(query)
        
        # --- Stage 1: FAISS retrieval (fetch more candidates for re-ranking) ---
        candidates_k = top_k * 4 if use_reranking else top_k
        
        query_embedding = self.embedder.encode(
            [cleaned_query], 
            normalize_embeddings=True
        ).astype('float32')
        
        scores, indices = self.index.search(query_embedding, candidates_k)
        
        candidates = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < len(self.documents) and score >= min_score:
                doc = self.documents[idx]
                
                # Apply character filter if specified
                if character_filter:
                    doc_char = doc.metadata.get('character', '').lower()
                    if character_filter.lower() not in doc_char:
                        continue
                
                candidates.append((doc, float(score)))
        
        if not candidates:
            return []
        
        # --- Stage 2: Re-rank with Historical BERT ---
        if use_reranking and len(candidates) > 1:
            query_bert_emb = self.extractor.create_embedding(
                cleaned_query[:500], method='mean'
            )
            
            reranked = []
            for doc, faiss_score in candidates:
                # Compute BERT similarity
                doc_bert_emb = self.extractor.create_embedding(
                    doc.text[:500], method='mean'
                )
                
                bert_sim = np.dot(query_bert_emb, doc_bert_emb) / (
                    np.linalg.norm(query_bert_emb) * 
                    np.linalg.norm(doc_bert_emb) + 1e-8
                )
                
                # Combined score
                combined = (1 - self.bert_weight) * faiss_score + \
                           self.bert_weight * bert_sim
                
                reranked.append((doc, float(combined)))
            
            reranked.sort(key=lambda x: x[1], reverse=True)
            return reranked[:top_k]
        
        return candidates[:top_k]
    
    def retrieve_for_character(
        self, 
        query: str, 
        character: str, 
        top_k: int = 5
    ) -> List[Tuple[HistoricalDocument, float]]:
        """Retrieve documents relevant to a specific historical character."""
        # Augment query with character context
        augmented_query = f"{character} {query}"
        return self.retrieve(
            augmented_query, 
            top_k=top_k, 
            character_filter=character
        )
    
    def format_context(
        self, 
        results: List[Tuple[HistoricalDocument, float]],
        max_context_length: int = 2000
    ) -> str:
        """Format retrieved documents into a context string for the LLM."""
        if not results:
            return "No relevant historical information found."
        
        context_parts = []
        total_length = 0
        
        for i, (doc, score) in enumerate(results, 1):
            meta = doc.metadata
            
            entry = (
                f"[Source {i} | Relevance: {score:.3f}]\n"
                f"Event: {meta.get('event_name', 'Unknown')}\n"
                f"Historical Figure: {meta.get('character', 'Unknown')}\n"
                f"Year: {meta.get('year', 'Unknown')}\n"
                f"Period: {meta.get('time_period', 'Unknown')}\n"
                f"Civilization: {meta.get('civilization', 'Unknown')}\n"
                f"Content: {doc.text}\n"
            )
            
            if total_length + len(entry) > max_context_length:
                break
            
            context_parts.append(entry)
            total_length += len(entry)
        
        return "\n---\n".join(context_parts)


# --- Instantiate Retriever ---
retriever = HistoricalRetriever(
    faiss_index=faiss_index,
    documents=documents,
    embedder=embedder,
    extractor=extractor,
    cleaner=cleaner,
    bert_rerank_weight=0.3
)

# Quick test
print("=" * 60)
print("🧪 RETRIEVAL TEST")
print("=" * 60)

test_queries = [
    "Tell me about the Punic Wars",
    "Who founded Carthage?",
    "What happened during the Ottoman period in Tunisia?",
]

for q in test_queries:
    results = retriever.retrieve(q, top_k=2, use_reranking=False)
    print(f"\n❓ Query: {q}")
    for doc, score in results:
        print(f"   → [{score:.3f}] {doc.metadata.get('event_name', 'N/A')[:60]}")
        print(f"     Character: {doc.metadata.get('character', 'N/A')}")

🧪 RETRIEVAL TEST

❓ Query: Tell me about the Punic Wars
   → [0.676] First Punic War Naval Defeat
     Character: Carthaginian Navy
   → [0.614] The Fall of Carthage
     Character: Hannibal Barca

❓ Query: Who founded Carthage?
   → [0.590] Phoenician Colonial Expansion
     Character: Dido (Elissa)
   → [0.581] Establishing Carthago Nova
     Character: Hamilcar Barca

❓ Query: What happened during the Ottoman period in Tunisia?
   → [0.740] Modernization Attempts
     Character: Ahmed Bey
   → [0.690] Ottoman Conquest
     Character: Hayreddin Barbarossa


In [11]:
# ============================================================================
# CELL 10: LOAD GENERATIVE LLM
# ============================================================================
"""
Part 5A: Load a generative language model for answer generation.

Model options (in order of preference for Kaggle):
  1. mistralai/Mistral-7B-Instruct-v0.2 (best quality, needs GPU)
  2. TinyLlama/TinyLlama-1.1B-Chat-v1.0 (lighter, still good)
  3. microsoft/phi-2 (good quality, medium size)
"""

print("=" * 60)
print("🤖 LOADING GENERATIVE LLM")
print("=" * 60)

# Try models in order of preference
LLM_CANDIDATES = [
    "TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    "microsoft/phi-2",
]

generator = None
llm_name = None

for model_name in LLM_CANDIDATES:
    try:
        print(f"\n📥 Attempting to load: {model_name}...")
        
        gen_tokenizer = AutoTokenizer.from_pretrained(model_name)
        
        # Set pad token if missing
        if gen_tokenizer.pad_token is None:
            gen_tokenizer.pad_token = gen_tokenizer.eos_token
        
        gen_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None,
            low_cpu_mem_usage=True
        )
        
        generator = pipeline(
            "text-generation",
            model=gen_model,
            tokenizer=gen_tokenizer,
            max_new_tokens=512,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
            repetition_penalty=1.2,
            pad_token_id=gen_tokenizer.pad_token_id
        )
        
        llm_name = model_name
        print(f"   ✅ {model_name} loaded successfully!")
        break
        
    except Exception as e:
        print(f"   ⚠️ Failed: {e}")
        continue

if generator is None:
    print("\n❌ Could not load any LLM. RAG answers will be context-only.")
else:
    print(f"\n✅ Active LLM: {llm_name}")

🤖 LOADING GENERATIVE LLM

📥 Attempting to load: TinyLlama/TinyLlama-1.1B-Chat-v1.0...


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Device set to use cuda:0


   ✅ TinyLlama/TinyLlama-1.1B-Chat-v1.0 loaded successfully!

✅ Active LLM: TinyLlama/TinyLlama-1.1B-Chat-v1.0


In [12]:
# ============================================================================
# CELL 11: HISTORICAL CHARACTER ROLE PROMPTS
# ============================================================================
"""
Part 5B: Define role prompts for multiple historical characters.

Each character has:
  - A persona description
  - Speaking style guidelines  
  - Strict instructions to use ONLY retrieved context
  - Anti-hallucination guardrails
"""

HISTORICAL_PERSONAS = {
    "hannibal barca": {
        "name": "Hannibal Barca",
        "title": "Supreme Commander of Carthage",
        "era": "3rd-2nd century BCE",
        "system_prompt": """You are Hannibal Barca, the legendary Carthaginian military commander 
who crossed the Alps with war elephants to challenge Rome. You speak in the first person as 
Hannibal himself.

YOUR CHARACTER TRAITS:
- You are brilliant, strategic, and proud of Carthage
- You speak with military precision and gravitas
- You reference your campaigns, battles, and the glory of Carthage
- You hold deep respect for worthy adversaries but burning hatred for Rome's treachery
- You refer to events you witnessed personally when relevant
- You speak of the Mediterranean as "our sea" and Africa as your homeland

STRICT RULES:
1. ONLY use information from the RETRIEVED CONTEXT provided below
2. If the context does not contain enough information, say "In my time, I did not witness this" 
   or "This falls beyond what I know from my campaigns"
3. NEVER invent battles, dates, or events not in the context
4. Always speak in FIRST PERSON as Hannibal
5. Weave the factual information naturally into your character's voice
6. If asked about events after your death (183 BCE), acknowledge you cannot speak to them 
   directly but may comment based on available context""",
    },
    
    "queen dido": {
        "name": "Queen Dido (Elissa)",
        "title": "Founder and Queen of Carthage",
        "era": "c. 814 BCE",
        "system_prompt": """You are Queen Dido, also known as Elissa, the legendary Phoenician 
princess who fled Tyre and founded the great city of Carthage. You speak in the first person.

YOUR CHARACTER TRAITS:
- You are wise, determined, and regal
- You speak with the dignity of a queen and the cleverness of a merchant princess
- You are proud of founding Carthage and the civilization it became
- You reference the story of the ox hide, your escape from Tyre, and the early days of Carthage
- You speak of the Phoenician heritage and maritime traditions
- You carry the sorrow of your brother Pygmalion's betrayal

STRICT RULES:
1. ONLY use information from the RETRIEVED CONTEXT provided below
2. If the context lacks information, say "The gods have not revealed this to me" 
   or "This is beyond the knowledge of my time"
3. NEVER invent facts not in the context
4. Always speak in FIRST PERSON as Queen Dido
5. Weave factual information into your regal, ancient voice""",
    },
    
    "ibn khaldun": {
        "name": "Ibn Khaldun",
        "title": "Father of Historiography and Sociology",
        "era": "14th century CE",
        "system_prompt": """You are Abd al-Rahman Ibn Khaldun, the great historian, scholar, and 
father of modern historiography and sociology. You wrote the Muqaddimah. You speak in first person.

YOUR CHARACTER TRAITS:
- You are analytical, philosophical, and deeply learned
- You speak with the wisdom of a scholar who has studied the rise and fall of civilizations
- You reference your theory of asabiyyah (social cohesion) and cyclical history
- You draw connections between events and broader historical patterns
- You are familiar with North African, Arab, and Mediterranean history
- You speak with measured, scholarly precision

STRICT RULES:
1. ONLY use information from the RETRIEVED CONTEXT provided below
2. If the context lacks information, say "My studies of history have not covered this matter" 
   or "The sources available to me do not speak to this"
3. NEVER invent historical facts not in the context
4. Always speak in FIRST PERSON as Ibn Khaldun
5. You may offer analytical commentary on patterns you observe in the provided facts""",
    },
    
    "habib bourguiba": {
        "name": "Habib Bourguiba",
        "title": "Father of Tunisian Independence",
        "era": "20th century CE",
        "system_prompt": """You are Habib Bourguiba, the leader of Tunisia's independence movement 
and the first President of the Republic of Tunisia. You speak in the first person.

YOUR CHARACTER TRAITS:
- You are charismatic, modernizing, and deeply patriotic
- You speak with the passion of a liberation leader and the vision of a nation-builder
- You reference your struggle against French colonialism
- You speak of education, women's rights, and building a modern secular state
- You are proud of the Personal Status Code and Tunisia's progressive reforms
- You speak with political eloquence and occasional dramatic flair

STRICT RULES:
1. ONLY use information from the RETRIEVED CONTEXT provided below
2. If the context lacks information, say "This was not part of our national struggle as I knew it"
3. NEVER invent political events or dates not in the context
4. Always speak in FIRST PERSON as Bourguiba
5. Connect facts to the broader narrative of Tunisian independence and modernization""",
    },
    
    "augustine of hippo": {
        "name": "Saint Augustine of Hippo",
        "title": "Bishop of Hippo, Theologian and Philosopher",
        "era": "4th-5th century CE",
        "system_prompt": """You are Augustine of Hippo, the great theologian, philosopher, and 
Bishop of Hippo Regius in Roman North Africa. You speak in the first person.

YOUR CHARACTER TRAITS:
- You are deeply reflective, philosophical, and spiritual
- You speak with the eloquence of a trained rhetorician
- You reference your Confessions, City of God, and your spiritual journey
- You are familiar with both Roman and early Christian North Africa
- You witnessed the decline of the Roman Empire and the Vandal invasions
- You blend philosophical reasoning with spiritual insight

STRICT RULES:
1. ONLY use information from the RETRIEVED CONTEXT provided below
2. If the context lacks information, say "This matter lies beyond my earthly knowledge"
3. NEVER invent facts not in the context
4. Always speak in FIRST PERSON as Augustine
5. You may reflect philosophically on the facts presented""",
    },
    
    "massinissa": {
        "name": "Massinissa",
        "title": "King of Numidia",
        "era": "3rd-2nd century BCE",
        "system_prompt": """You are Massinissa, King of Numidia, who united the Numidian tribes 
and allied with Rome against Carthage. You speak in the first person.

YOUR CHARACTER TRAITS:
- You are proud, ambitious, and a skilled horseman and warrior
- You speak of the Numidian people, their heritage, and their lands
- You reference your alliance with Scipio and the defeat of Carthage
- You are proud of building Numidia into a powerful kingdom
- You speak of agriculture, urbanization, and the prosperity you brought
- You carry the complexity of choosing Rome over Carthage

STRICT RULES:
1. ONLY use information from the RETRIEVED CONTEXT provided below
2. If the context lacks information, say "The sands of Numidia hold no memory of this"
3. NEVER invent facts not in the context
4. Always speak in FIRST PERSON as Massinissa
5. Weave factual information into your warrior-king voice""",
    },
}

print("✅ Historical personas defined:")
for key, persona in HISTORICAL_PERSONAS.items():
    print(f"   🎭 {persona['name']} — {persona['title']} ({persona['era']})")

✅ Historical personas defined:
   🎭 Hannibal Barca — Supreme Commander of Carthage (3rd-2nd century BCE)
   🎭 Queen Dido (Elissa) — Founder and Queen of Carthage (c. 814 BCE)
   🎭 Ibn Khaldun — Father of Historiography and Sociology (14th century CE)
   🎭 Habib Bourguiba — Father of Tunisian Independence (20th century CE)
   🎭 Saint Augustine of Hippo — Bishop of Hippo, Theologian and Philosopher (4th-5th century CE)
   🎭 Massinissa — King of Numidia (3rd-2nd century BCE)


In [13]:
# ============================================================================
# CELL 12: RAG ANSWER GENERATOR
# ============================================================================
"""
Part 5C: The complete RAG pipeline — retrieval + role-prompted generation.

Flow:
  1. User asks a question and selects a historical character
  2. Retriever finds relevant documents
  3. Context + role prompt are combined
  4. LLM generates an in-character answer constrained to retrieved facts
"""

class HistoricalRAGSystem:
    """Complete RAG system with historical character role-playing."""
    
    def __init__(
        self,
        retriever: HistoricalRetriever,
        generator,
        personas: Dict,
        cleaner: HistoricalTextCleaner,
        extractor: HistoricalEntityExtractor
    ):
        self.retriever = retriever
        self.generator = generator
        self.personas = personas
        self.cleaner = cleaner
        self.extractor = extractor
    
    def _detect_character(self, query: str) -> Optional[str]:
        """Auto-detect which character the user might be asking about."""
        query_lower = query.lower()
        
        character_keywords = {
            'hannibal barca': [
                'hannibal', 'barca', 'alps', 'punic war', 'elephants', 'zama',
                'cannae', 'trebia', 'trasimene', 'carthaginian general'
            ],
            'queen dido': [
                'dido', 'elissa', 'founding of carthage', 'phoenician princess',
                'ox hide', 'tyre', 'pygmalion'
            ],
            'ibn khaldun': [
                'ibn khaldun', 'khaldun', 'muqaddimah', 'historiography',
                'asabiyyah', 'social cohesion', 'sociology'
            ],
            'habib bourguiba': [
                'bourguiba', 'independence', 'tunisian republic', 'neo-destour',
                'protectorate', 'decolonization', 'personal status'
            ],
            'augustine of hippo': [
                'augustine', 'hippo', 'confessions', 'city of god',
                'bishop', 'theologian', 'christian north africa'
            ],
            'massinissa': [
                'massinissa', 'masinissa', 'numidia', 'numidian',
                'numidian king', 'berber king'
            ],
        }
        
        best_match = None
        best_count = 0
        
        for char_key, keywords in character_keywords.items():
            count = sum(1 for kw in keywords if kw in query_lower)
            if count > best_count:
                best_count = count
                best_match = char_key
        
        return best_match if best_count > 0 else None
    
    def _build_prompt(
        self, 
        query: str, 
        context: str, 
        character_key: str
    ) -> str:
        """Build the full prompt with role instructions and retrieved context."""
        persona = self.personas[character_key]
        
        prompt = (
            f"### SYSTEM ###\n"
            f"{persona['system_prompt']}\n\n"
            f"### RETRIEVED HISTORICAL CONTEXT ###\n"
            f"The following information has been retrieved from verified historical sources.\n"
            f"Use ONLY this information to answer the question. "
            f"Do not add any facts not present here.\n\n"
            f"{context}\n\n"
            f"### END OF CONTEXT ###\n\n"
            f"### USER QUESTION ###\n"
            f"{query}\n\n"
            f"### RESPONSE (as {persona['name']}) ###\n"
        )
        return prompt
    
    def answer(
        self,
        query: str,
        character_key: Optional[str] = None,
        top_k: int = 5,
        max_context_length: int = 2000,
        verbose: bool = True
    ) -> Dict[str, Any]:
        """
        Generate a historically-grounded, in-character answer.
        
        Args:
            query: User's question
            character_key: Which character should answer (auto-detected if None)
            top_k: Number of documents to retrieve
            max_context_length: Max chars for context
            verbose: Print intermediate steps
        
        Returns:
            Dictionary with answer, sources, character info
        """
        result = {
            'query': query,
            'character': None,
            'answer': '',
            'sources': [],
            'entities_detected': [],
            'retrieval_scores': []
        }
        
        # --- Step 1: Detect or set character ---
        if character_key is None:
            character_key = self._detect_character(query)
        
        if character_key is None:
            character_key = 'ibn khaldun'
            if verbose:
                print("   ℹ️  No specific character detected, defaulting to Ibn Khaldun")
        
        if character_key not in self.personas:
            character_key = 'ibn khaldun'
        
        persona = self.personas[character_key]
        result['character'] = persona['name']
        
        if verbose:
            print(f"   🎭 Speaking as: {persona['name']} — {persona['title']}")
        
        # --- Step 2: Extract entities from query ---
        entities = self.extractor.extract_entities(query)
        result['entities_detected'] = [
            {'text': e['text'], 'type': e['type']} for e in entities
        ]
        
        # ✅ FIX: Build the entity string separately to avoid f-string escaping issues
        if verbose and entities:
            meaningful = [e for e in entities if e['type'] != 'OTHER']
            if meaningful:
                entity_strings = []
                for e in meaningful[:5]:
                    entity_strings.append("{}({})".format(e['text'], e['type']))
                joined = ', '.join(entity_strings)
                print(f"   🔍 Entities: {joined}")
        
        # --- Step 3: Retrieve relevant documents ---
        if verbose:
            print(f"   📚 Retrieving top-{top_k} documents...")
        
        retrieved = self.retriever.retrieve(
            query, top_k=top_k, use_reranking=True
        )
        
        if not retrieved:
            no_info_responses = {
                'hannibal barca': "I, Hannibal, have marched across many lands, yet the records of my scouts contain nothing on this matter.",
                'queen dido': "The gods have not revealed this knowledge to me, and no Phoenician scroll speaks of it.",
                'ibn khaldun': "My extensive studies of history and civilization have not uncovered information on this topic.",
                'habib bourguiba': "In all my years of struggle for our nation, this matter did not cross my path.",
                'augustine of hippo': "Neither my earthly studies nor divine illumination have granted me knowledge of this.",
                'massinissa': "The winds of Numidia carry no whisper of this matter to my ears.",
            }
            result['answer'] = no_info_responses.get(
                character_key,
                f"As {persona['name']}, I must confess that the historical records "
                f"available to me contain no information on this matter."
            )
            return result
        
        result['sources'] = [
            {
                'event': doc.metadata.get('event_name', 'Unknown'),
                'character': doc.metadata.get('character', 'Unknown'),
                'year': doc.metadata.get('year', 'Unknown'),
                'score': float(score),
                'preview': doc.text[:100]
            }
            for doc, score in retrieved
        ]
        result['retrieval_scores'] = [s for _, s in retrieved]
        
        if verbose:
            best_score = retrieved[0][1]
            print(f"   📄 Retrieved {len(retrieved)} documents (best score: {best_score:.3f})")
        
        # --- Step 4: Format context ---
        context = self.retriever.format_context(
            retrieved, max_context_length=max_context_length
        )
        
        # --- Step 5: Generate answer ---
        prompt = self._build_prompt(query, context, character_key)
        
        if self.generator is not None:
            if verbose:
                print(f"   🤖 Generating response with {llm_name}...")
            
            try:
                output = self.generator(
                    prompt,
                    max_new_tokens=400,
                    do_sample=True,
                    temperature=0.7,
                    top_p=0.9,
                    repetition_penalty=1.2
                )
                
                generated = output[0]['generated_text']
                
                # Extract only the response part
                response_marker = "### RESPONSE (as {}) ###".format(persona['name'])
                if response_marker in generated:
                    answer = generated.split(response_marker)[-1].strip()
                else:
                    answer = generated[len(prompt):].strip()
                
                # Clean up any trailing prompt artifacts
                cleanup_markers = [
                    '### SYSTEM', '### RETRIEVED', 
                    '### USER', '### END', '###'
                ]
                for marker in cleanup_markers:
                    if marker in answer:
                        answer = answer[:answer.index(marker)].strip()
                
                # Remove any repeated lines
                lines = answer.split('\n')
                seen = set()
                unique_lines = []
                for line in lines:
                    stripped = line.strip()
                    if stripped and stripped not in seen:
                        seen.add(stripped)
                        unique_lines.append(line)
                answer = '\n'.join(unique_lines)
                
                result['answer'] = answer if answer else self._fallback_answer(
                    persona, context, query
                )
                
            except Exception as e:
                if verbose:
                    print(f"   ⚠️ Generation error: {e}")
                result['answer'] = self._fallback_answer(persona, context, query)
        else:
            result['answer'] = self._fallback_answer(persona, context, query)
        
        return result
    
    def _fallback_answer(
        self, persona: Dict, context: str, query: str
    ) -> str:
        """Generate a structured answer when LLM is unavailable."""
        
        # Character-specific intro phrases
        intros = {
            'Hannibal Barca': "I, Hannibal, commander of Carthage's armies, shall tell you what I know.",
            'Queen Dido (Elissa)': "As the founder of great Carthage, I shall share what the records reveal.",
            'Ibn Khaldun': "As a student of history and civilization, allow me to present what the sources show.",
            'Habib Bourguiba': "As the leader who fought for Tunisia's freedom, I share these historical facts.",
            'Saint Augustine of Hippo': "In my years of study and reflection, these are the truths I have found.",
            'Massinissa': "As King of the Numidians, I recall these events from the annals of our people.",
        }
        
        name = persona['name']
        intro = intros.get(name, "Based on the historical records available to me:")
        
        return (
            f"Speaking as {name}, {persona['title']}:\n\n"
            f"{intro}\n\n"
            f"{context}\n\n"
            f"[Note: This is a direct context retrieval. "
            f"LLM generation was not available for in-character narration.]"
        )


# --- Instantiate the full system ---
rag_system = HistoricalRAGSystem(
    retriever=retriever,
    generator=generator,
    personas=HISTORICAL_PERSONAS,
    cleaner=cleaner,
    extractor=extractor
)

print("✅ Historical RAG System initialized")
llm_status = 'Yes - ' + llm_name if generator else 'No (context-only mode)'
print(f"   LLM available: {llm_status}")
print(f"   Characters available: {len(HISTORICAL_PERSONAS)}")
for key, persona in HISTORICAL_PERSONAS.items():
    print(f"      🎭 {persona['name']}")

✅ Historical RAG System initialized
   LLM available: Yes - TinyLlama/TinyLlama-1.1B-Chat-v1.0
   Characters available: 6
      🎭 Hannibal Barca
      🎭 Queen Dido (Elissa)
      🎭 Ibn Khaldun
      🎭 Habib Bourguiba
      🎭 Saint Augustine of Hippo
      🎭 Massinissa


In [14]:
# ============================================================================
# CELL 13: COMPREHENSIVE Q&A TESTING — MULTIPLE CHARACTERS
# ============================================================================
"""
Part 6A: Test the full RAG pipeline with questions answered by 
different historical characters. Each character responds in their 
unique voice, constrained to retrieved historical facts.
"""

print("=" * 70)
print("🏛️  HISTORICAL RAG Q&A — MULTI-CHARACTER DEMONSTRATION")
print("=" * 70)

# --- Test Questions with Designated Characters ---
test_scenarios = [
    {
        "question": "Tell me about the Battle of Zama and its significance",
        "character": "hannibal barca",
        "description": "Hannibal reflects on his greatest defeat"
    },
    {
        "question": "How was the city of Carthage founded?",
        "character": "queen dido",
        "description": "Queen Dido tells the story of founding Carthage"
    },
    {
        "question": "What caused the rise and fall of great North African civilizations?",
        "character": "ibn khaldun",
        "description": "Ibn Khaldun analyzes historical cycles"
    },
    {
        "question": "How did Tunisia achieve independence from France?",
        "character": "habib bourguiba",
        "description": "Bourguiba narrates the independence struggle"
    },
    {
        "question": "What was life like in Roman North Africa?",
        "character": "augustine of hippo",
        "description": "Augustine reflects on Roman African society"
    },
    {
        "question": "Tell me about the Numidian kingdoms and their role in history",
        "character": "massinissa",
        "description": "Massinissa speaks of Numidian glory"
    },
    {
        "question": "What were the Punic Wars about?",
        "character": "hannibal barca",
        "description": "Hannibal explains the conflict with Rome"
    },
    {
        "question": "How did Islamic civilization arrive in North Africa?",
        "character": "ibn khaldun",
        "description": "Ibn Khaldun on the Arab conquest"
    },
]

for i, scenario in enumerate(test_scenarios, 1):
    print(f"\n{'='*70}")
    print(f"📜 TEST {i}: {scenario['description']}")
    print(f"{'='*70}")
    print(f"❓ Question: {scenario['question']}")
    print(f"🎭 Character: {HISTORICAL_PERSONAS[scenario['character']]['name']}")
    print("-" * 70)
    
    result = rag_system.answer(
        query=scenario['question'],
        character_key=scenario['character'],
        top_k=4,
        verbose=True
    )
    
    print(f"\n💬 ANSWER ({result['character']}):")
    print("-" * 40)
    # Wrap text for readability
    wrapped = textwrap.fill(result['answer'], width=80, initial_indent="   ", 
                           subsequent_indent="   ")
    print(wrapped)
    
    if result['sources']:
        print(f"\n📚 Sources ({len(result['sources'])} documents retrieved):")
        for j, src in enumerate(result['sources'][:3], 1):
            print(f"   {j}. {src['event'][:50]} | Score: {src['score']:.3f}")
            print(f"      Figure: {src['character']} | Year: {src['year']}")
    
    print()

🏛️  HISTORICAL RAG Q&A — MULTI-CHARACTER DEMONSTRATION

📜 TEST 1: Hannibal reflects on his greatest defeat
❓ Question: Tell me about the Battle of Zama and its significance
🎭 Character: Hannibal Barca
----------------------------------------------------------------------
   🎭 Speaking as: Hannibal Barca — Supreme Commander of Carthage
   🔍 Entities: significance(DATE)
   📚 Retrieving top-4 documents...
   📄 Retrieved 4 documents (best score: 0.612)
   🤖 Generating response with TinyLlama/TinyLlama-1.1B-Chat-v1.0...

💬 ANSWER (Hannibal Barca):
----------------------------------------
   Thank you for asking about the Battle of Zama! The Battle of Zama was an
   important moment in Roman history, as it marked the end of Hannibal's initial
   invasion of Italy. Hannibal faced Scipio Africanus, one of the most
   experienced commanders of his time, during the Battle of Zama. Despite being
   outnumbered and outmatched, Hannibal managed to win the battle using his
   elephant army's superio

In [15]:
# ============================================================================
# CELL 14: INTERACTIVE Q&A SESSION
# ============================================================================
"""
Part 6B: Interactive loop where users can ask questions and choose 
which historical character should answer.
"""

def interactive_session():
    """Run an interactive Q&A session with historical characters."""
    
    print("=" * 70)
    print("🏛️  INTERACTIVE HISTORICAL Q&A SESSION")
    print("=" * 70)
    print("\nAvailable historical characters:")
    
    char_list = list(HISTORICAL_PERSONAS.keys())
    for i, (key, persona) in enumerate(HISTORICAL_PERSONAS.items(), 1):
        print(f"   {i}. {persona['name']} — {persona['title']} ({persona['era']})")
    
    print(f"   0. Auto-detect (system chooses the best character)")
    print(f"\nType 'quit' to exit.\n")
    
    while True:
        print("-" * 50)
        query = input("❓ Your question: ").strip()
        
        if query.lower() in ['quit', 'exit', 'q', '']:
            print("\n👋 Farewell, seeker of historical wisdom!")
            break
        
        # Character selection
        print("\nChoose a character (number or name, 0 for auto):")
        char_input = input("🎭 Character: ").strip()
        
        character_key = None
        
        if char_input == '0' or char_input.lower() == 'auto':
            character_key = None  # Auto-detect
        elif char_input.isdigit():
            idx = int(char_input) - 1
            if 0 <= idx < len(char_list):
                character_key = char_list[idx]
            else:
                print("   ⚠️ Invalid number, using auto-detect")
        else:
            # Try to match by name
            char_lower = char_input.lower()
            for key in char_list:
                if char_lower in key or char_lower in HISTORICAL_PERSONAS[key]['name'].lower():
                    character_key = key
                    break
        
        print()
        result = rag_system.answer(
            query=query,
            character_key=character_key,
            top_k=5,
            verbose=True
        )
        
        print(f"\n{'='*60}")
        print(f"🎭 {result['character']} speaks:")
        print(f"{'='*60}")
        wrapped = textwrap.fill(result['answer'], width=75, 
                               initial_indent="   ", subsequent_indent="   ")
        print(wrapped)
        
        if result['sources']:
            print(f"\n📚 Based on {len(result['sources'])} historical sources")
            for j, src in enumerate(result['sources'][:3], 1):
                print(f"   {j}. {src['event'][:50]} (Score: {src['score']:.3f})")
        
        print()

# Run interactive session
#interactive_session() #juste eliminated the ## in this line

In [16]:
# ============================================================================
# CELL 15: SYSTEM EVALUATION & METRICS
# ============================================================================
"""
Part 6C: Evaluate the RAG system's retrieval quality and coverage.
"""

print("=" * 70)
print("📊 SYSTEM EVALUATION & STATISTICS")
print("=" * 70)

# --- Document Coverage ---
print("\n📄 DOCUMENT STORE STATISTICS:")
print(f"   Total documents: {len(documents)}")
print(f"   FAISS index size: {faiss_index.ntotal}")
print(f"   Embedding dimension: {EMBED_DIM}")

# Character distribution
char_dist = defaultdict(int)
period_dist = defaultdict(int)
civ_dist = defaultdict(int)

for doc in documents:
    char_dist[doc.metadata.get('character', 'Unknown')] += 1
    period_dist[doc.metadata.get('time_period', 'Unknown') or 'Unknown'] += 1
    civ_dist[doc.metadata.get('civilization', 'Unknown') or 'Unknown'] += 1

print(f"\n👤 Character Distribution (top 10):")
for char, count in sorted(char_dist.items(), key=lambda x: -x[1])[:10]:
    print(f"   {char:30s}: {count:4d} documents")

print(f"\n⏰ Time Period Distribution:")
for period, count in sorted(period_dist.items(), key=lambda x: -x[1]):
    print(f"   {str(period):20s}: {count:4d} documents")

print(f"\n🏛️  Civilization Distribution:")
for civ, count in sorted(civ_dist.items(), key=lambda x: -x[1]):
    print(f"   {str(civ):20s}: {count:4d} documents")

# --- Retrieval Quality Test ---
print(f"\n🔍 RETRIEVAL QUALITY TEST:")
eval_queries = [
    ("Battle of Zama", "hannibal barca"),
    ("Founding of Carthage", "queen dido"),
    ("Tunisian independence", "habib bourguiba"),
    ("Muqaddimah", "ibn khaldun"),
    ("Roman Africa", "augustine of hippo"),
    ("Numidian kingdom", "massinissa"),
]

for query, expected_char in eval_queries:
    results = retriever.retrieve(query, top_k=3, use_reranking=False)
    
    if results:
        top_doc, top_score = results[0]
        top_char = top_doc.metadata.get('character', 'Unknown').lower()
        char_match = expected_char.lower() in top_char or top_char in expected_char.lower()
        
        status = "✅" if char_match else "⚠️"
        print(f"   {status} '{query}' → Score: {top_score:.3f} | "
              f"Got: {top_doc.metadata.get('character', 'N/A')[:25]} | "
              f"Expected: {expected_char}")
    else:
        print(f"   ❌ '{query}' → No results")

# --- Model Info ---
print(f"\n🤖 MODEL CONFIGURATION:")
print(f"   Historical BERT: {HISTORICAL_BERT} (active: {USE_HISTORICAL})")
print(f"   Embedder: {EMBED_MODEL} (dim: {EMBED_DIM})")
print(f"   LLM: {llm_name if llm_name else 'None (context-only mode)'}")
print(f"   Vector Store: FAISS IndexFlatIP")
print(f"   Device: {device}")

print(f"\n{'='*70}")
print("✅ HISTORICAL RAG SYSTEM — FULLY OPERATIONAL")
print(f"{'='*70}")

📊 SYSTEM EVALUATION & STATISTICS

📄 DOCUMENT STORE STATISTICS:
   Total documents: 208
   FAISS index size: 208
   Embedding dimension: 768

👤 Character Distribution (top 10):
   Hannibal Barca                :   20 documents
   Habib Bourguiba               :   17 documents
   Masinissa                     :    8 documents
   Zine El Abidine Ben Ali       :    7 documents
   Hanno The Great               :    6 documents
   Hamilcar Barca                :    4 documents
   Hanno The Navigator           :    4 documents
   Dido (Elissa)                 :    4 documents
   Ubayd Allah Al-Mahdi          :    4 documents
   Ahmed Bey                     :    4 documents

⏰ Time Period Distribution:
   modern              :  111 documents
   ancient             :   63 documents
   colonial            :   20 documents
   medieval            :    8 documents
   late antiquity      :    4 documents
   early modern        :    2 documents

🏛️  Civilization Distribution:
   carthaginian        